In [1]:
5-1

4

In [2]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
import seaborn as sns
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm


import ast
import re

# import nltk
# from nltk.stem import WordNetLemmatizer


# Task - Save a csv file with prompts for SAM3 model to segment.

-- Come up with csv file containing which prompts to run for each video:

- Video —----- prompt1 — prompt2 — prompt3
- Vid1.mp4 — knife —----- tomato —- cuttingboard


In [3]:
ssv2_path = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/public/ssv2"
labels_dir = Path(ssv2_path) / "labels"

# SSv2 annotations use train.json and validation.json
train_file = labels_dir / "train.json"
val_file = labels_dir / "validation.json"

with open(train_file, 'r') as f:
    train_raw = json.load(f)

with open(val_file, 'r') as f:
    val_raw = json.load(f)

train_raw[0:2] # train_raw is a list. i am printing first 2 elements which are training examples.

[{'id': '78687',
  'label': 'holding potato next to vicks vaporub bottle',
  'template': 'Holding [something] next to [something]',
  'placeholders': ['potato', 'vicks vaporub bottle']},
 {'id': '42326',
  'label': 'spreading margarine onto bread',
  'template': 'Spreading [something] onto [something]',
  'placeholders': ['margarine', 'bread']}]

In [4]:
df_train = pd.DataFrame(train_raw)
df_train['split'] = 'train'

df_val = pd.DataFrame(val_raw)
df_val['split'] = 'val'

df = pd.concat([df_train, df_val], ignore_index=True)
df.head(2)

,id,label,template,placeholders,split
0,78687,holding potato next to vicks vaporub bottle,Holding [something] next to [something],"[potato, vicks vaporub bottle]",train
1,42326,spreading margarine onto bread,Spreading [something] onto [something],"[margarine, bread]",train


In [17]:
df.placeholders.apply(len).value_counts()

placeholders
1    111881
2     78565
3      3244
Name: count, dtype: int64

In [6]:
pd.set_option('display.width', 10000)
print(df.head(2))

      id                                        label                                 template                    placeholders  split
0  78687  holding potato next to vicks vaporub bottle  Holding [something] next to [something]  [potato, vicks vaporub bottle]  train
1  42326               spreading margarine onto bread   Spreading [something] onto [something]              [margarine, bread]  train


In [12]:
# Expand list elements into columns and slice the first 3
expanded = df["placeholders"].apply(pd.Series).iloc[:, :3]

# Rename columns to c1, c2, c3
expanded.columns = [f"c{i+1}" for i in range(expanded.shape[1])]

# Pad with None if the list had fewer than 3 elements in total
for col in ["c1", "c2", "c3"]:
    if col not in expanded.columns:
        expanded[col] = None

# Join back to original DataFrame
df = pd.concat([df, expanded[["c1", "c2", "c3"]]], axis=1)

df.head(4)

,id,label,template,placeholders,split,c1,c2,c3
0,78687,holding potato next to vicks vaporub bottle,Holding [something] next to [something],"[potato, vicks vaporub bottle]",train,potato,vicks vaporub bottle,NaN
1,42326,spreading margarine onto bread,Spreading [something] onto [something],"[margarine, bread]",train,margarine,bread,NaN
2,100904,putting pen on a surface,Putting [something] on a surface,[pen],train,pen,NaN,NaN
3,80715,"lifting up one end of bottle, then letting it ...","Lifting up one end of [something], then lettin...",[bottle],train,bottle,NaN,NaN


In [16]:
df[df.placeholders.apply(len)>2].head(2)

,id,label,template,placeholders,split,c1,c2,c3
53,97012,failing to put led bulb box into glass bowl be...,Failing to put [something] into [something] be...,"[led bulb box, glass bowl, glass bowl]",train,led bulb box,glass bowl,glass bowl
268,75650,"putting sticky note, binder clip and pen on th...","Putting [something], [something] and [somethin...","[sticky note, binder clip, pen]",train,sticky note,binder clip,pen


In [19]:
df.columns = ['id', 'label', 'template', 'placeholders', 'split', 'prompt1', 'prompt2', 'prompt3']
df.head(1)

,id,label,template,placeholders,split,prompt1,prompt2,prompt3
0,78687,holding potato next to vicks vaporub bottle,Holding [something] next to [something],"[potato, vicks vaporub bottle]",train,potato,vicks vaporub bottle,NaN


In [ ]:
# df.to_csv('./ssv2_videos_with_prompts_for_sam3.csv', index=False)